# Titulo del proyecto: Análisis y modelado de ingresos mediante técnicas de minería de datos


Por: Ana Milena Ibarbo Gutierrez









# Objetivo de la mineria:

*	Desarrollar un m**odelo descriptivo tipo Clustering** para agrupar a la población en segmentos con características socioeconómicas similares, mediante técnicas de agrupamiento no supervisado sobre variables del dataset Adults.









# Ciclo de vida para Clustering

1. Preparación de datos: variables numéricas se deben normalizar y variables categóricas se crean dummies
2. Aprendizaje del Modelo: Kmeans, método del codo
3. Evaluación del Modelo: Inertia, silueta
4. Perfilamiento: Descripción de centroides



# Valor esperado de la evaluación:

* Se espera identificar entre 2 y 5 grupos poblacionales claramente diferenciados según sus características socioeconómicas, con un índice de silueta superior a 0.5 que indique buena separación entre grupos.









**Carga de librerias**

In [ ]:
#Importamos las librerias requeridas para el desarrollo del Modelo
#=================================================================
# Tratamiento de datos

import pandas as pd #manipular dataframe
import matplotlib.pyplot as plt #graficos
import numpy as np #matrices y vectores
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler

#Modelado
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 1. Preparación y Carga de Datos



La base de datos fue procesada en Weka para hacer la limpieza de atípicos, nulos y altas correlaciones; se ubicó en una ruta comuún para la carga y desarrollo de los modelos de acuerdo con la asignacion de cada integrante y corresponden a daatos libres disponibles en Kaggle (https://www.kaggle.com/datasets/uciml/adult-census-income)


In [ ]:
path = "https://github.com/camilojoanbarrancogalvis-debug/colab/raw/refs/heads/main/adult_clean.csv"
df = pd.read_csv(path)
df.head()

In [ ]:
#Revisión del tipo de datos cargados
df.info()

In [ ]:
#Corrección de tipo de datos
for col in df.columns.tolist():
  if df[col].dtype != np.int64:
    df[col] = df[col].astype('category')
df.info()

In [ ]:
#Descripcion estadistica de variables numericas.
df.describe()

In [ ]:
#Histogramas de las variables numericas
df.hist()

In [ ]:
#Graficos box para variables numericas
df.plot(kind='box')

In [ ]:
#Reagrupacion de variables para transformaciones

data = df.copy()
numericas = ['age', 'capital-gain', 'capital-loss', 'hours-per-week']
categoricas_binarias = ['sex' ,'income']
categoricas_narias = ['workclass', 'marital-status', 'education', 'occupation']

In [ ]:
#Graficas de variables categoricas
#Grafica de Clase
data['sex'].value_counts().plot(kind='bar')

In [ ]:
data['income'].value_counts().plot(kind='bar')

In [ ]:
data['workclass'].value_counts().plot(kind='bar')

In [ ]:
data['education'].value_counts().plot(kind='bar')

In [ ]:
data['marital-status'].value_counts().plot(kind='bar')

In [ ]:
data['occupation'].value_counts().plot(kind='bar')

In [ ]:
#Analisis exploratorio de datos- intalación libreria
!pip install ydata-profiling

In [ ]:
#Analisis descriptivo
from ydata_profiling import ProfileReport
profile_data = ProfileReport(df)
ProfileReport(df)

In [ ]:
#Guardamos en html el perfilado de datos
profile_data.to_file(output_file="output.html")

**BALANCEO DE DATOS: NO REQUERIDO POR EL TIPO DE ANALISIS QUE SE DESARROLLA**

In [ ]:
#Copia de datos previo a las transformaciones
data = df.copy()

# Ingeniería de Características



In [ ]:
#normalizacion de variables numericas
from sklearn.preprocessing import MinMaxScaler
min_max_scaler = MinMaxScaler()
numericas = ['age', 'capital-gain', 'capital-loss', 'hours-per-week']
data[numericas] = min_max_scaler.fit_transform(data[numericas])
data.head()

In [ ]:
# Ahora aplicamos get_dummies para las categóricas
# Dummies para variables narias
categoricas_narias = ['workclass', 'marital-status', 'education', 'occupation']
cols_to_dummy = [col for col in categoricas_narias if col in data.columns]

if cols_to_dummy:
    data = pd.get_dummies(data, columns=cols_to_dummy, drop_first=False, dtype=int)

# Dummies para variables binarias (sex, income)
if 'sex' in data.columns:
    data = pd.get_dummies(data, columns=['sex'], drop_first=True, dtype=int)
if 'income' in data.columns:
    data = pd.get_dummies(data, columns=['income'], drop_first=True, dtype=int)

data.head()

# 2. Aprendizaje del Modelo
* Metodo del codo
* Metodo de la rodilla
* Aplicar Kmeans

In [ ]:
# ============================================================
# COMPARAR MÚLTIPLES K: MÉTODO DEL CODO + SILHOUETTE
# ============================================================

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn import metrics
import numpy as np # Import numpy for argmax

k_range = range(2, 20)
inercias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data)

    inercias.append(kmeans.inertia_)
    silhouettes.append(metrics.silhouette_score(data, kmeans.labels_))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(k_range, inercias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo')
axes[0].grid(True)
axes[0].set_xticks(k_range) # Set x-ticks to all values in k_range

# Silhouette
axes[1].plot(k_range, silhouettes, 'go-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette vs K')
axes[1].axhline(y=0.5, color='r', linestyle='--', label='Umbral 0.5')
axes[1].legend()
axes[1].grid(True)
axes[1].set_xticks(k_range) # Set x-ticks to all values in k_range

plt.tight_layout()
plt.show()

# Encontrar K óptimo según cada métrica
print(f"K óptimo por Silhouette: {list(k_range)[np.argmax(silhouettes)]}")

Miremos como son los resultados con lo que vimos en la clase!!!!

In [ ]:
#Método del codo para encontrar la mejor cantidad de clusters: inertia
from sklearn.cluster import KMeans

ks = range(2, 20) # crear valores del 2 al 20
inertias = []

for k in ks:
    # Crear  modelo
    model = KMeans(n_clusters=k,max_iter=300)
    model.fit(data)
    inertias.append(model.inertia_)

# Graficar cantidad de clusters vs inertias
plt.plot(ks, inertias, '-o')
plt.xlabel('Numero de clusters, k')
plt.ylabel('inertia')
plt.xticks(ks)
plt.show()

In [ ]:
#Método de la rodilla: silueta
from sklearn import metrics

ks = range(2, 20) # crear valores del 2 al 20
siluetas = []

for k in ks:
    # Crear  modelo
    model = KMeans(n_clusters=k,max_iter=300)
    model.fit(data)
    sil=metrics.silhouette_score(data, model.labels_)
    siluetas.append(sil)

# Graficar cantidad de clusters vs inertias
plt.plot(ks, siluetas, '-o')
plt.xlabel('Numero de clusters, k')
plt.ylabel('silueta')
plt.xticks(ks)
plt.show()

Se crea el modelo de Kmeans con el cluster que cuenta con la medida de silueta mas alta que en este caso es 3.

In [ ]:
#Creación de modelo de clustering con Kmeans
from sklearn.cluster import KMeans
k=3
model = KMeans(n_clusters=k, max_iter=300)
model.fit(data) #100% datos

# 3. Evaluación del Modelo


*   Inertia: medidas intra cluster- valor pequeño esperado
*   Silueta: valor positivo esperado, idealmente mayor a 0.5



In [ ]:
#Evaluación
from sklearn import metrics

#Inertia: se require valor pequeño
print('Inercia o cohesión:', model.inertia_)

#Silueta: se requiere que sea positivo, ideal 0.5-1.0
sil=metrics.silhouette_score(data, model.labels_)
print('Silueta:',sil)

Con los resultados obtenidos de las mediciones de la inercia se revisa bibliografia de referencia: https://scikit-learn.org/stable/modules/clustering.html, aca se indica:


" **Inertia is not a normalized metric**: **we just know that lower values are better and zero is optima**l. But in very high-dimensional spaces, Euclidean distances tend to become inflated (this is an instance of the so-called “curse of dimensionality”). **Running a dimensionality reduction algorithm such as Principal component analysis (PCA) prior to k-means clustering can alleviate this problem and speed up the computations**."

Resaltado fuera de texto






# 4. Perfilamiento

Descripción de centroides

In [ ]:
#Centroides de los cluster se convierten  en un dataframe de pandas
centroides=pd.DataFrame(model.cluster_centers_, columns=data.columns.values)
centroides.round(1)

In [ ]:
#Se realiza des-normalizacion de centroides
centroides[numericas]=min_max_scaler.inverse_transform(centroides[numericas])
# Display the centroides with two decimal places for better readability and interpretation
display(centroides.round(2))

**DESCRIPCIÓN DE PERFILES**

**PERFIL 0:** Trabajadores estables del sector público: "funcionarios y trabajadores independientes de clase media-baja: estables, solteros o divorciados, con horario fijo y sin ambiciones de grandes fortunas."

Son personas con trabajo estable, pero sin grandes aspiraciones de riqueza. Muchos eligieron el sector público buscando seguridad laboral, beneficios y horarios predecibles. No suelen invertir en bolsa ni tener ganancias de capital —su dinero viene del sueldo mensual.

Una buena parte nunca se casó o está divorciada, lo que sugiere vidas más independientes o con menos presión social tradicional. Trabajan lo necesario, no suelen hacer horas extras, y su nivel educativo les permite mantenerse cómodamente, pero sin lujos.

**PERFIL 1**:Ejecutivos casados con Éxito: Hombres casados de mediana edad en el sector privado: trabajan mucho, ganan bien, y representan el arquetipo tradicional de éxito profesional de los años 90."

Son el grupo de mayor éxito económico. La gran mayoría son hombres casados (90%), trabajan en empresas privadas, ocupan cargos de responsabilidad o especialización técnica, y dedican más horas al trabajo que los otros grupos.

El hecho de que casi todos estén casados sugiere estabilidad familiar, posiblemente doble ingreso (esposa también trabaja), y una red social que facilita oportunidades laborales. Tienen más educación formal, lo que les abrió puertas a mejores puestos.
Tienen algo de inversión en capital (ganancias y pérdidas de capital ligeramente mayores), lo que indica que algunos tienen bienes raíces, acciones o negocios paralelos.

Este grupo refleja la desigualdad de género de la época (1994-1996). No significa que los hombres sean inherentemente mejores ejecutivos, sino que en ese momento histórico tenían más acceso a estos puestos. Usar esto para decisiones actuales sin cuestionar el sesgo sería un error.


**PERFIL 2:** Los jóvenes del sector privado en desarrollo: Jóvenes trabajadores del sector privado en sus primeros pasos laborales: solteros, con poca experiencia, bajos ingresos, y un futuro aún por definir.

Son el grupo más joven y de menor ingreso. Todos trabajan en empresas privadas, probablemente en sus primeros empleos o en posiciones junior. Muchos están solteros, posiblemente viviendo solos, con compañeros, o con padres.
Sus ocupaciones son variadas y muchas veces transitorias: ventas, servicios, oficina, oficios técnicos. No han consolidado una carrera todavía. Trabajan menos horas que los demás, quizás porque tienen empleos de medio tiempo, están estudiando paralelamente, o simplemente no tienen la senioridad para exigir más responsabilidades (y horas).
Tienen casi nula inversión en capital —no tienen dinero sobrante para invertir en bolsa o bienes raíces.


In [ ]:
#En el dataframe original, se adiciona el cluster asignado a cada registro
df['cluster']=model.labels_
df.head()

In [ ]:
#Cantidad de datos en cada cluster
df["cluster"].value_counts().plot(kind='pie',autopct='%.0f%%')

In [ ]:
#Almacenar resultados
df.to_excel('./resultados_Kmeans.xlsx')
centroides.to_excel('./centroides.xlsx')
